# Feature process
Turn str features into hard-coded
Separate a features per 4h
Merge vital signs and lab results according to the hamd_id

In [ ]:
import pandas as pd
import os
from pathlib import Path
import numpy as np

In [ ]:
raw_dir = r"data/MIMICIII_last48h"
primary_dir = f"data/MIMICIII_last48h_ts2h"

# keep the last n rows according time_resolution, making the total time window 48h
time_resolution = "2h"
if time_resolution == "2h":
    num_timestep = 24
    time_resolution_sec = 2*3600
elif time_resolution == "1h":
    num_timestep = 48
    time_resolution_sec = 3600
elif time_resolution == "30min":
    num_timestep = 96
    time_resolution_sec = 30*60
else:
    raise ValueError(f"Unknown time resolution {time_resolution}")

primary_timeseries_dir = os.path.join(primary_dir, "timeseries")
timeseries_dir = f"data/MIMICIII_last48h_ts{time_resolution}/timeseries"
Path(timeseries_dir).mkdir(exist_ok=True, parents=True)

## ICD9 Count and occurrence

In [ ]:
# ICD9 occurrence
diag_path = os.path.join(raw_dir, 'diag.csv')
diag_export_path = os.path.join(timeseries_dir, 'diag.csv')
icd9_export_path = os.path.join(timeseries_dir, 'icd9.csv')

diag_df = pd.read_csv(diag_path, dtype={'icd9_code': str})
diag_df['icd9_code'] = diag_df['icd9_code'].apply(lambda x: 'ICD_' + x[:3])

diag_df.to_csv(diag_export_path, index=False, sep=',')
diag_df.info()

In [ ]:
# icd9 count
icd9_counts = diag_df['icd9_code'].value_counts()

icd9_df = icd9_counts.reset_index()
icd9_df.columns = ['icd9_code', 'count']

icd9_df.to_csv(icd9_export_path, index=False, sep=',')

icd9_df.info()
used_icd9_list = icd9_df['icd9_code'].tolist()

## Drug

In [ ]:
# Set paths
drug_path = os.path.join(raw_dir, 'drug.csv')
drug_export_path = os.path.join(timeseries_dir, 'drug.csv')

# Read CSV file
drug_df = pd.read_csv(drug_path)

# Process icd9_code column and filter desired icd9_code
drug_df['icd9_code'] = drug_df['icd9_code'].apply(lambda x: 'ICD_' + x[:3])
drug_df = drug_df[drug_df["icd9_code"].isin(used_icd9_list)]

# Extract hadm_id, drug, icd9_code columns and drop duplicates
unique_drug_data = drug_df[['hadm_id', 'drug', 'icd9_code']].drop_duplicates()

# Output results to CSV file
unique_drug_data.to_csv(drug_export_path, index=False)

# Print data info
unique_drug_data.info()

## Demographics

In [ ]:
demo_path = os.path.join(raw_dir, 'demographics.csv')
demo_export_path = os.path.join(timeseries_dir, 'demographics.csv')

demo_df = pd.read_csv(demo_path, dtype={'icd9_code': str})
demo_df['icd9_code'] = demo_df['icd9_code'].apply(lambda x: 'ICD_' + x[:3])

demo_df = demo_df[demo_df['icd9_code'].isin(used_icd9_list)]
considered_hadm_ids = demo_df['hadm_id'].unique()

demo_df.to_csv(demo_export_path,index=False, sep=',')
demo_df.info()

## Vital signs

In [ ]:
vital_path = os.path.join(raw_dir, 'vital.csv')
vital_export_path = os.path.join(timeseries_dir, 'vital.csv')

vital_df = pd.read_csv(vital_path)
vital_df = vital_df[vital_df['hadm_id'].isin(considered_hadm_ids)]

# change chartime to timepoints
vital_df['timepoint'] = vital_df['charttime']//time_resolution_sec
vital_df['timepoint'] = vital_df['timepoint'].astype('int')

# get max timepoints
max_timepoints = vital_df.groupby('hadm_id')['timepoint'].max().reset_index()
max_timepoints.columns = ['hadm_id', 'max_timepoint']

# generate timepoints
def generate_timepoints(row, n=num_timestep):
    return pd.DataFrame({
        'hadm_id': [row['hadm_id']] * n,
        'timepoint': list(range(int(row['max_timepoint'] - n + 1), int(row['max_timepoint'] + 1)))
    })

lastn_timepoints = pd.concat(max_timepoints.apply(generate_timepoints, axis=1).tolist()).reset_index(drop=True)

# merge
vital_df = pd.merge(lastn_timepoints, vital_df, on=['hadm_id', 'timepoint'], how='left')
vital_df = vital_df.sort_values(['hadm_id', 'timepoint'])

# 对每个 hadm_id，把 timepoint 映射到 0..n-1
vital_df['timepoint'] = (
    vital_df['timepoint']
    - vital_df.groupby('hadm_id')['timepoint'].transform('max')
    + (num_timestep - 1)
).astype(int)

vital_df.head()

#### Calculate variability metrics per timepoint and across timepoints

In [ ]:
# -----------------------------
# 1) Column setup
# -----------------------------
numeric_cols = vital_df.select_dtypes(include=[np.number]).columns.tolist()
id_cols = ['hadm_id', 'timepoint']
feature_cols = [c for c in numeric_cols if c not in id_cols]

for aux_col in ('subject_id', 'charttime'):
    if aux_col in feature_cols:
        feature_cols.remove(aux_col)

# -----------------------------
# 2) Define custom funcs
# -----------------------------
def _cv(x: pd.Series):
    m = x.mean()
    return np.nan if m == 0 or np.isnan(m) else x.std(ddof=1) / m

# -----------------------------
# 3) Groupby aggregate once
# -----------------------------
g = vital_df.groupby(['timepoint'], sort=False, observed=True)

agg_dict = {
    col: [('mean', 'mean'),
          ('std', 'std'),
          ('min', 'min'),
          ('max', 'max'),
          ('count', 'count'),
          ('cv', _cv)]
    for col in feature_cols
}

if 'subject_id' in vital_df.columns:
    agg_dict['subject_id'] = [('first', 'first')]
if 'charttime' in vital_df.columns:
    agg_dict['charttime'] = [('mean', 'mean')]

agg = g.agg(agg_dict)  # MultiIndex columns: (feature, stat)

# -----------------------------
# 4) Derive global means from the aggregated result
# -----------------------------
# Select all (feature, 'std') and (feature, 'cv') columns and compute row-wise mean
std_cols = [col for col in agg.columns if col[1] == 'std']
cv_cols  = [col for col in agg.columns if col[1] == 'cv']

# Compute per-group global means (skip NaN)
global_std_mean = agg.loc[:, std_cols].mean(axis=1, skipna=True)
global_cv_mean  = agg.loc[:, cv_cols].mean(axis=1, skipna=True)

# Attach as new single-level columns on the same index
agg[('GLOBAL', 'all_features_std_mean')] = global_std_mean
agg[('GLOBAL', 'all_features_cv_mean')]  = global_cv_mean

# -----------------------------
# 5) Flatten columns and finalize
# -----------------------------
out = agg.reset_index()
out.columns = [f"{a}_{b}" if b else f"{a}" for a, b in out.columns]

# Drop charttime aggregation if not needed
drop_cols = [c for c in out.columns if c.startswith('charttime_')]
if drop_cols:
    out.drop(columns=drop_cols, inplace=True)

# -----------------------------
# 6) Save
# -----------------------------
out_path = os.path.join(timeseries_dir, 'vital_PerTimepoint_stats.csv')
out.to_csv(out_path, index=False)


In [ ]:
vital_last48hours_df = vital_df.groupby(['hadm_id', 'timepoint']).mean().reset_index()
vital_last48hours_df.drop(['charttime'], axis=1, inplace=True)

def last_n_rows(group, n=num_timestep):
    if len(group) >= n:
        return group.tail(n)
    else:
        timepoints = range(n-len(group), n)  # new timepoint series
        return group.assign(timepoint=timepoints)

vital_last48hours_df = vital_last48hours_df.groupby('hadm_id').apply(last_n_rows).reset_index(drop=True)
vital_last48hours_df['timepoint'] = vital_last48hours_df.groupby('hadm_id').cumcount()

vital_last48hours_df.to_csv(vital_export_path, index=False, sep=',')
vital_last48hours_df.head()

In [ ]:
vital_last48hours_df["timepoint"].value_counts()

## Lab Results

In [ ]:
lab_path = os.path.join(raw_dir, 'labs.csv')
lab_export_path = os.path.join(timeseries_dir, 'labs.csv')

raw_lab_df = pd.read_csv(lab_path)
raw_lab_df = raw_lab_df[raw_lab_df['hadm_id'].isin(considered_hadm_ids)]

labs = ['ANION GAP', 'ALBUMIN', 'BANDS', 'BICARBONATE', 'BILIRUBIN', 'CREATININE', 'CHLORIDE', 'HEMATOCRIT', 'HEMOGLOBIN', 'LACTATE', 'PLATELET', 'POTASSIUM', 'PTT', 'INR', 'PT', 'SODIUM', 'BUN', 'WBC']
raw_lab_df['timepoint'] = raw_lab_df['charttime']//time_resolution_sec
raw_lab_df['timepoint'].astype('int')

raw_lab_df.info()

In [ ]:
# 1) compute timepoint (make sure it is cast to int)
raw_lab_df['timepoint'] = (raw_lab_df['charttime'] // time_resolution_sec).astype('int')

# if you want to keep only the lab tests of interest (replace 'label' with the column name for test names in your table)
# raw_lab_df = raw_lab_df[raw_lab_df['label'].isin(labs)]

# 2) max timepoint per hadm_id (aligned row-wise with the original table)
gmax = raw_lab_df.groupby('hadm_id')['timepoint'].transform('max')

# 3) take the last num_timestep steps
mask = raw_lab_df['timepoint'] >= (gmax - (num_timestep - 1))
lab_lastn = raw_lab_df[mask].copy()

# ——up to here, lab_lastn contains the last num_timestep steps for each hadm (possibly with gaps, and possibly multiple rows per test)

# (optional) 4) re-index timepoint to 0..num_timestep-1 for easier alignment/modeling
lab_lastn['timepoint'] = (lab_lastn['timepoint'] - gmax[mask] + (num_timestep - 1)).astype('int')

In [ ]:
# -----------------------------
# labs: Per-(stay_id,timepoint) statistics table
# Output style consistent with vital_PerTimepoint_stats.csv
# -----------------------------
# 1) Define coefficient of variation (CV)
def _cv(x: pd.Series):
    m = x.mean()
    return np.nan if m == 0 or np.isnan(m) else x.std(ddof=1) / m

# 2) Group by (stay_id, timepoint, label) and aggregate valuenum
#    Use observed=True to reduce Cartesian expansion of categorical groups
g = lab_lastn.groupby(['timepoint', 'label'], sort=False, observed=True)['valuenum']

agg = g.agg(
    mean='mean',
    std=lambda s: s.std(ddof=1),
    min='min',
    max='max',
    count='count',
    cv=_cv
)

# 3) Promote labname from row index to column, multi-level columns: (stat, labname)
#    Then swap column levels to match vital code structure: (feature=labname, stat)
agg = agg.unstack('label')                  # columns: (stat, labname)
agg = agg.swaplevel(0, 1, axis=1)             # columns: (labname, stat)
agg = agg.sort_index(axis=1)                  # Optional: tidy column grouping

# 4) Compute "global" means (row-wise mean of std/cv across all lab features)
std_cols = [col for col in agg.columns if col[1] == 'std']
cv_cols  = [col for col in agg.columns if col[1] == 'cv']

global_std_mean = agg.loc[:, std_cols].mean(axis=1, skipna=True)
global_cv_mean  = agg.loc[:, cv_cols].mean(axis=1, skipna=True)

agg[('GLOBAL', 'all_features_std_mean')] = global_std_mean
agg[('GLOBAL', 'all_features_cv_mean')]  = global_cv_mean

# 5) Flatten column names and finalize
out = agg.reset_index()

# Consistent flattening with vital code: (feature, stat) -> "feature_stat"
# Note: after reset_index, index columns appear as ('stay_id',''), ('timepoint',''), handled as single level
out.columns = [f"{a}_{b}" if b else f"{a}" for a, b in out.columns]

# labs table has no charttime aggregation columns, so no need to drop; if added later, reuse same drop logic
# drop_cols = [c for c in out.columns if c.startswith('charttime_')]
# if drop_cols:
#     out.drop(columns=drop_cols, inplace=True)

# 6) Save
lab_stats_path = os.path.join(timeseries_dir, 'lab_PerTimepoint_stats.csv')
out.to_csv(lab_stats_path, index=False)


In [ ]:
lab_df = raw_lab_df.pivot_table(
    index=['hadm_id', 'timepoint'], 
    columns='label', 
    values='valuenum', 
    aggfunc='mean'
).reset_index()

max_timepoints = lab_df.groupby('hadm_id')['timepoint'].max().reset_index()
max_timepoints.columns = ['hadm_id', 'max_timepoint']

lastn_timepoints = pd.concat(max_timepoints.apply(generate_timepoints, axis=1).tolist()).reset_index(drop=True)

lab_df = pd.merge(lastn_timepoints, lab_df, on=['hadm_id', 'timepoint'], how='left')

lab_df.head()

# generate lab_last48hours_df from lab_df
lab_last48hours_df = lab_df.groupby('hadm_id').apply(last_n_rows).reset_index(drop=True)
lab_last48hours_df['timepoint'] = lab_last48hours_df.groupby('hadm_id').cumcount()

lab_last48hours_df.to_csv(lab_export_path, index=False, sep=',')
lab_last48hours_df.head()

In [ ]:
lab_last48hours_df["timepoint"].value_counts()

# Merge vital signs and lab results according to the hadm_id

In [ ]:
demo_df = pd.read_csv(os.path.join(timeseries_dir, 'demographics.csv'))
vital_df = pd.read_csv(os.path.join(timeseries_dir, 'vital.csv'))
lab_df = pd.read_csv(os.path.join(timeseries_dir, 'labs.csv'))

ts_df = pd.merge(vital_df, lab_df, on=['hadm_id', 'timepoint'], how='outer')
used_hadm_ids = demo_df['hadm_id'].unique()

ts_df = ts_df[ts_df['hadm_id'].isin(used_hadm_ids)]
included_hadm_id = ts_df['hadm_id'].unique()
demo_df = demo_df[demo_df['hadm_id'].isin(included_hadm_id)]
assert demo_df['hadm_id'].unique().shape[0] == ts_df['hadm_id'].unique().shape[0]

demo_df.to_csv(demo_export_path, index=False, sep=",")
ts_df.to_csv(os.path.join(timeseries_dir, 'time-series.csv'), index=False, sep=',')
ts_df["timepoint"].value_counts()

# Labels
Includes the 30-day-readmissions and 90-day-mortality

In [ ]:
readmission_csv_path = os.path.join(raw_dir, 'readmission.csv')
mortality_csv_path = os.path.join(raw_dir, 'mortality.csv')
label_export_csv_path = os.path.join(timeseries_dir, 'label.csv')

readmission_df = pd.read_csv(readmission_csv_path)
mortality_df = pd.read_csv(mortality_csv_path)

label_df = pd.merge(readmission_df, mortality_df, on='hadm_id')
label_df.rename(columns={'subject_id_x': 'subject_id'}, inplace=True)
label_df.drop(['subject_id_y'], axis=1, inplace=True)

label_df.to_csv(label_export_csv_path, index=False)
label_df.info()